## car prediction project using mutivariable linear regression with feature engenniring and feature scaling and gradient decent

In [33]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler   


## 1 load dataset

In [3]:
df = pd.read_csv("archive(1)/car_price_prediction_.csv")
print(df.shape)
df.head()

(2500, 10)


,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,2,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,3,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,4,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,5,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


## 2.process data :we can notice that some of our data is not int/float so the solution i came wit is using something called One-Hot Encoding 

the concept of this method is turning string into digits for example i can make fuel types each of them a feature and say either 1 or 0

now we can check which features should i apply this method on 

In [4]:
for col in ['Brand', 'Fuel Type', 'Transmission', 'Condition']:
    print(col, ":", df[col].unique())

Brand : <StringArray>
['Tesla', 'BMW', 'Audi', 'Ford', 'Honda', 'Mercedes', 'Toyota']
Length: 7, dtype: str
Fuel Type : <StringArray>
['Petrol', 'Electric', 'Diesel', 'Hybrid']
Length: 4, dtype: str
Transmission : <StringArray>
['Manual', 'Automatic']
Length: 2, dtype: str
Condition : <StringArray>
['New', 'Used', 'Like New']
Length: 3, dtype: str


In [5]:
df = pd.get_dummies(df, columns=['Brand', 'Fuel Type', 'Transmission', 'Condition'], drop_first=True)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
print(df.shape)
df.head()

(2500, 18)


,Car ID,Year,Engine Size,Mileage,Price,Model,Brand_BMW,Brand_Ford,Brand_Honda,Brand_Mercedes,Brand_Tesla,Brand_Toyota,Fuel Type_Electric,Fuel Type_Hybrid,Fuel Type_Petrol,Transmission_Manual,Condition_New,Condition_Used
0,1,2016,2.3,114832,26613.92,Model X,0,0,0,0,1,0,0,0,1,1,1,0
1,2,2018,4.4,143190,14679.61,5 Series,1,0,0,0,0,0,1,0,0,1,0,1
2,3,2013,4.5,181601,44402.61,A4,0,0,0,0,0,0,1,0,0,1,1,0
3,4,2011,4.1,68682,86374.33,Model Y,0,0,0,0,1,0,0,0,0,0,1,0
4,5,2009,2.6,223009,73577.10,Mustang,0,1,0,0,0,0,0,0,0,1,0,0


## now we can see that the problem is mostly solved we have unused features like model and id and the year feature can be turned into smaller values by deviding it from the current year

In [6]:
df['Car Age'] = 2025 - df['Year']
df = df.drop(columns=['Car ID', 'Model', 'Year'])
print(df.shape)
df.head()

(2500, 16)


,Engine Size,Mileage,Price,Brand_BMW,Brand_Ford,Brand_Honda,Brand_Mercedes,Brand_Tesla,Brand_Toyota,Fuel Type_Electric,Fuel Type_Hybrid,Fuel Type_Petrol,Transmission_Manual,Condition_New,Condition_Used,Car Age
0,2.3,114832,26613.92,0,0,0,0,1,0,0,0,1,1,1,0,9
1,4.4,143190,14679.61,1,0,0,0,0,0,1,0,0,1,0,1,7
2,4.5,181601,44402.61,0,0,0,0,0,0,1,0,0,1,1,0,12
3,4.1,68682,86374.33,0,0,0,0,1,0,0,0,0,0,1,0,14
4,2.6,223009,73577.10,0,1,0,0,0,0,0,0,0,1,0,0,16


## now we notice that some features have big values while other have small ones so we need feature scalling we have to choose between : max scalling, mean normalization and z-score scalling

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Price'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_cols = ['Mileage', 'Engine Size', 'Car Age']

a practical way to scale feature in the whole dataset is by spliting it 

In [8]:
def max_scaling(X_train, X_test, cols):
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    
    max_vals = X_train[cols].max()
    
    X_train_scaled[cols] = X_train[cols] / max_vals
    X_test_scaled[cols] = X_test[cols] / max_vals  #
    
    return X_train_scaled, X_test_scaled, max_vals

In [9]:
def mean_normalization(X_train, X_test, cols):
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    
    mean_vals = X_train[cols].mean()
    min_vals = X_train[cols].min()
    max_vals = X_train[cols].max()
    range_vals = max_vals - min_vals
    
    X_train_scaled[cols] = (X_train[cols] - mean_vals) / range_vals
    X_test_scaled[cols] = (X_test[cols] - mean_vals) / range_vals
    
    return X_train_scaled, X_test_scaled, (mean_vals, range_vals)

In [10]:
def z_score_normalization(X_train, X_test, cols):
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    
    mean_vals = X_train[cols].mean()
    std_vals = X_train[cols].std()
    
    X_train_scaled[cols] = (X_train[cols] - mean_vals) / std_vals
    X_test_scaled[cols] = (X_test[cols] - mean_vals) / std_vals
    
    return X_train_scaled, X_test_scaled, (mean_vals, std_vals)

In [11]:
def sklearn_scaling(X_train, X_test, cols):
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    
    scaler = StandardScaler()
    X_train_scaled[cols] = scaler.fit_transform(X_train[cols])
    X_test_scaled[cols] = scaler.transform(X_test[cols])
    
    return X_train_scaled, X_test_scaled, scaler

sklearn method is per default the same as z-score

In [12]:
X_train_max, X_test_max, _ = max_scaling(X_train, X_test, numeric_cols)
X_train_meannorm, X_test_meannorm, _ = mean_normalization(X_train, X_test, numeric_cols)
X_train_zscore, X_test_zscore, _ = z_score_normalization(X_train, X_test, numeric_cols)
X_train_sklearn, X_test_sklearn, _ = sklearn_scaling(X_train, X_test, numeric_cols)

z-score and sklearn should give near scores

In [13]:
print(X_train_zscore[numeric_cols].head())
print(X_train_sklearn[numeric_cols].head())

       Mileage  Engine Size   Car Age
2055 -0.882925    -0.040959  0.939792
1961  1.153074    -1.302316  1.083414
1864 -0.695290     1.640851  1.370659
2326  1.416055     0.799946  1.657903
461  -1.249644    -0.181110  0.221681
       Mileage  Engine Size   Car Age
2055 -0.883145    -0.040969  0.940027
1961  1.153363    -1.302642  1.083685
1864 -0.695464     1.641261  1.371001
2326  1.416409     0.800146  1.658318
461  -1.249957    -0.181155  0.221736


# now after scalling data we will apply gradient decent but first we convert the data into numpy arrays

In [14]:
X_train_np = X_train_zscore.values
X_test_np = X_test_zscore.values
y_train_np = y_train.values
y_test_np = y_test.values

In [35]:
def compute_cost(X, w, b, Y):
    m = X.shape[0]
    predictions = X @ w + b
    cost = np.sum((predictions - Y) ** 2) / (2 * m)
    return cost

In [21]:
w_init = np.zeros(X_train_zscore.shape[1])
b_init = 0.0

cost = compute_cost(X_train_zscore.values, w_init, b_init, y_train.values)
print(cost)
print(np.mean(y_train.values ** 2) / 2)

1746820298.452649
1746820298.4526513


now gradient decent 

In [36]:
def compute_gradient(X, w, b, Y):
    m = X.shape[0]
    predictions = X @ w + b
    errors = predictions - Y
    dj_dw = (X.T @ errors) / m
    dj_db = np.sum(errors) / m
    return dj_dw, dj_db

# test

In [23]:
dj_dw, dj_db = compute_gradient(X_train_zscore.values, w_init, b_init, y_train.values)
print(dj_db)
print(-y_train.mean())

-52461.688634999984
-52461.688635000006


## now the gradient descent function

In [37]:
def gradient_descent(X, Y, w_init, b_init, learning_rate, n_iterations):
    w = w_init.copy()
    b = b_init
    cost_history = []
    
    for i in range(n_iterations):
        dj_dw, dj_db = compute_gradient(X, w, b, Y)
        
        w = w - learning_rate * dj_dw
        b = b - learning_rate * dj_db
        
        cost = compute_cost(X, w, b, Y)
        cost_history.append(cost)
        
        if i % 100 == 0:
            print(f"Iteration {i}: Cost = {cost:.2f}")
    
    return w, b, cost_history

## i want to try it on the three scalling methods as an experiment

In [38]:
def run_gd(X, y, learning_rate, n_iterations=1000):
    w_init = np.zeros(X.shape[1])
    b_init = 0.0
    w, b, cost_history = gradient_descent(X, y, w_init, b_init, learning_rate, n_iterations)
    return cost_history

# this plot is interactive to see how the convergance is affected by learning rate 

In [40]:
def plot_convergence(learning_rate):
    n_iterations = 500
    other_cols = [c for c in X_train.columns if c not in numeric_cols]
    feature_cols = numeric_cols + other_cols

    methods = {
        "Max Scaling": X_train_max[feature_cols].values,
        "Mean Normalization": X_train_meannorm[feature_cols].values,
        "Z-score Normalization": X_train_zscore[feature_cols].values,
    }

    colors = {"Max Scaling": "tab:blue", "Mean Normalization": "tab:orange", "Z-score Normalization": "tab:green"}

    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

    for ax, (name, X_data) in zip(axes, methods.items()):
        w, b, cost_history = gradient_descent(
            X_data, y_train.values,
            np.zeros(X_data.shape[1]), 0.0,
            learning_rate, n_iterations
        )

        final_cost = cost_history[-1]
        initial_cost = cost_history[0]
        pct_reduction = 100 * (initial_cost - final_cost) / initial_cost

        # detect divergence
        diverged = np.isnan(final_cost) or final_cost > initial_cost

        ax.plot(cost_history, color=colors[name], linewidth=2)
        ax.set_title(f"{name}\nlr={learning_rate:.4g}", fontsize=11)
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Cost (MSE/2)")
        ax.grid(alpha=0.3)

        if diverged:
            ax.text(0.5, 0.5, "DIVERGED", color="red", fontsize=16, fontweight="bold",
                     ha='center', va='center', transform=ax.transAxes,
                     bbox=dict(facecolor='white', edgecolor='red'))
        else:
            # annotate final cost + % reduction on the plot
            ax.annotate(
                f"Final cost: {final_cost:,.0f}\nReduction: {pct_reduction:.1f}%",
                xy=(0.98, 0.95), xycoords='axes fraction',
                ha='right', va='top', fontsize=9,
                bbox=dict(facecolor='white', edgecolor='gray', alpha=0.8)
            )

        # log scale toggle if cost varies over orders of magnitude — makes flattening visible
        if not diverged and final_cost > 0 and initial_cost / max(final_cost, 1e-9) > 50:
            ax.set_yscale('log')

    plt.suptitle(f"Gradient Descent Convergence Across Scaling Methods (learning_rate={learning_rate:.4g})", fontsize=13)
    plt.tight_layout()
    plt.show()

interact(
    plot_convergence,
    learning_rate=FloatLogSlider(value=0.01, base=10, min=-4, max=1, step=0.1, description='Learning rate')
)

interactive(children=(FloatLogSlider(value=0.01, description='Learning rate', max=1.0, min=-4.0), Output()), _…

<function __main__.plot_convergence(learning_rate)>